# NewFiguresAug26 — plots

Reads the `.npz` files written by `NewFiguresAug26_calc.ipynb` and renders Figures 1–5
of `docs/figureDescription_Aug26.md`. **No physics here** — if a panel is empty, run the
matching section of the calc notebook.

Global toggles (top cell):
* `POOL` — `"targeted"` or `"general"` — drives Fig 3b and Fig 5d only.
  **Figure 2 c/d/g/h are always the full converged *targeted* pool** (`fig2c_targeted`,
  `fig2d_targeted`), independent of this toggle.
* `NOCOMP_HESSIAN` — `"unconstrained"` or `"constrained"` — which lowest eigenvalue the
  *no-compression* series uses in Fig 2c/2g.
* `FIG2A_HESSIAN` — `"constrained"` or `"unconstrained"` — which rest spectrum Fig 2a/2e shows.

Figure 2 layout:
* **a / e** — eigenvalues of the elastic Hessian of the network **at rest** vs `L/L_init`
  (constrained spectrum), tracked through training for the representative task.
* **b / f** — constrained elastic spectrum vs actuation strain; colour = training state
  (grey before / red best-loss), line style = subtask ramp, legend labels it by the
  imposed strain.
* **c / g** — distribution of `λ₀(after) / λ₀(before)` (lowest non-trivial elastic-Hessian
  eigenvalue, best-loss K over initial K) at 3 compressions (no compression / subtask 1 /
  subtask 2), all targeted realizations, series labelled by imposed strain. Log x, dashed
  line at ratio 1.
* **d / h** — histogram of the actuation-trajectory position of diabatic avoided
  crossings, before vs after training, pooled over all targeted realizations.

In [ ]:
import importlib, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

sys.path.insert(0, str(Path.cwd()))
import aug26_common as C
importlib.reload(C)

# ---------------- GLOBAL TOGGLES ----------------
POOL = "targeted"                 # "targeted" | "general"  -- Fig 3b / Fig 5d only
NOCOMP_HESSIAN = "unconstrained"  # "unconstrained" | "constrained"  -- Fig 2c/2g no-compression curve
FIG2A_HESSIAN = "constrained"     # "constrained" | "unconstrained"  -- which rest spectrum Fig 2a/2e shows
FIG2_POOL = "targeted"            # Fig 2 c/d/g/h are fixed to the full targeted pool
SAVE_PDF = True
OUT = C.REPO_ROOT / "figure_data" / "aug26_figs"

plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "font.size": 10})

def get(kind, name):
    try:
        return C.load_fig_data(kind, name)
    except FileNotFoundError:
        print(f"  [missing] {kind}/{name}.npz — run the calc notebook section")
        return None

def savefig(fig, stem):
    if SAVE_PDF:
        p = OUT / f"{stem}.pdf"
        fig.savefig(p)
        print("saved", p)

## Figure 1 — networks & response

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.suptitle("Figure 1 — trained networks and input/output response")

def panel_network(ax, dd, title):
    if dd is None:
        ax.set_axis_off(); return
    lc = C.draw_network(ax, dd["nodes"], dd["edges"], dd["stiffnesses"], cmap="magma")
    cb = fig.colorbar(lc, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label(r"$\log_{10} k$")
    ax.set_title(title)

panel_network(axes[0, 0], get("allo", "fig1_allo_network"), "a — allosteric (targeted), edges ~ stiffness")
panel_network(axes[1, 0], get("aux", "fig1_aux_network"),  "c — auxetic (targeted), edges ~ stiffness")

# ---- 1b : allosteric free-run response, one curve per task ----
# The plotted curve is the *deep* subtask (subtask 1): input node pulled to
# eps_in = strain_input, output strain read off the free (unclamped) relaxation.
# Markers show the (input, output) pairs each network was *trained* for:
#   * filled star  -> subtask 1 target  (strain_input,  strain_output)   -- lies on the curve
#   * open circle  -> subtask 2 target  (strain_input2, strain_output2)  -- a different, shorter
#                     pull, so it is NOT expected to sit on the subtask-1 curve
d = get("allo", "fig1_allo_response")
ax = axes[0, 1]
if d is not None:
    for cu in d["curves"]:
        line, = ax.plot(cu["in_frac"] * cu["strain_input"], cu["out_strain"], lw=1.5,
                        label=f"task {cu['ids'][1]}")
        col = line.get_color()
        ax.scatter([cu["strain_input"]], [cu["target"]], marker="*", s=130, zorder=6,
                   color=col, edgecolors="k", linewidths=0.5)
        if "target2" in cu:
            ax.scatter([cu["strain_input2"]], [cu["target2"]], marker="o", s=42, zorder=6,
                       facecolors="none", edgecolors=col, linewidths=1.4)
    handles = [Line2D([0], [0], marker="*", color="k", ls="none", ms=13, mfc="0.7"),
               Line2D([0], [0], marker="o", color="k", ls="none", ms=7, mfc="none")]
    leg1 = ax.legend(handles, ["subtask 1 target", "subtask 2 target"],
                     fontsize=7, loc="lower left")
    ax.add_artist(leg1)
    ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.set_xlabel(r"input strain $\varepsilon_{\rm in}$"); ax.set_ylabel(r"output strain $\varepsilon_{\rm out}$")
ax.set_title("b — allosteric response, 5 tasks / one geometry")

# ---- 1d : auxetic response vs |applied strain|, one curve per task ----
# Curve = deepest subtask's compression (method='fire', so nu[-1] matches the
# recomputed subtask loss). The first array entry is the undeformed frame
# (eps_yy = 0 -> nu forced to 0), so it is dropped. Markers = every subtask's
# trained (|compression|, nu_target):
#   * filled star -> subtask 1 (deepest, endpoint of the curve)
#   * open circle -> shallower subtasks (their own shorter compression)
d = get("aux", "fig1_aux_response")
ax = axes[1, 1]
if d is not None:
    for cu in d["curves"]:
        line, = ax.plot(np.abs(cu["eps_yy"][1:]), cu["nu"][1:], lw=1.5, label=f"task {cu['ids'][0]}")
        col = line.get_color()
        sub_cs = cu["sub_compression"] if "sub_compression" in cu else np.array([cu["compression"]])
        sub_tp = cu["sub_target"]      if "sub_target"      in cu else np.array([cu["target"]])
        for si, (csi, tpi) in enumerate(zip(np.abs(sub_cs), sub_tp)):
            if si == 0:
                ax.scatter([csi], [tpi], marker="*", s=130, zorder=6,
                           color=col, edgecolors="k", linewidths=0.5)
            else:
                ax.scatter([csi], [tpi], marker="o", s=42, zorder=6,
                           facecolors="none", edgecolors=col, linewidths=1.4)
    handles = [Line2D([0], [0], marker="*", color="k", ls="none", ms=13, mfc="0.7"),
               Line2D([0], [0], marker="o", color="k", ls="none", ms=7, mfc="none")]
    leg1 = ax.legend(handles, ["subtask 1 target", "subtask 2 target"], fontsize=7, loc="lower left")
    ax.add_artist(leg1)
    ax.legend(fontsize=7, ncol=2, loc="upper right")
ax.set_xlabel(r"applied strain magnitude $|\varepsilon_{yy}|$"); ax.set_ylabel(r"Poisson ratio $\nu$")
ax.set_title("d — auxetic response, 5 tasks / one geometry")

fig.tight_layout()
savefig(fig, "fig1_aug26")
plt.show()

## Figure 2 — Hessian structure (a–d allosteric, e–h auxetic)

* **a / e** rest-configuration elastic-Hessian spectrum vs `L/L_init` (tracked through training)
* **b / f** constrained elastic spectrum vs the **actual imposed strain**, for the **deepest
  subtask only** (its ramp already spans the shallower subtask's range). Two curve sets:
  grey = initial K, red = best-loss K. Dotted line = where the shallow subtask's ramp ends.
* **c / g** distribution of `λ₀(after)/λ₀(before)` at 3 compressions — **all converged
  targeted realizations**, series labelled by imposed strain, stacked
* **d / h** actuation-trajectory position of diabatic avoided crossings, before vs after
  training — histogram pooled over **all converged targeted realizations**

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle(f"Figure 2 - rest Hessian={FIG2A_HESSIAN}, no-compression Hessian={NOCOMP_HESSIAN}; "
             f"c/d/g/h = full targeted pool")


def _strain_label(arr, sym):
    """'<sym> = v' when the strains are all equal, else '<sym> in {v1, v2}'."""
    a = np.asarray(arr, float).ravel() if arr is not None else np.array([])
    a = a[np.isfinite(a)]
    if len(a) == 0:
        return None
    u = np.unique(np.round(a, 3))
    if len(u) == 1:
        return rf"${sym} = {u[0]:g}$"
    return rf"${sym} \in \{{{', '.join(f'{x:g}' for x in u)}\}}$"

# ---- 2a / 2e : elastic Hessian of the network AT REST vs L/L_init ----
def plot_2a(ax, d, n_modes=24, which=None):
    if d is None:
        return
    which = which or FIG2A_HESSIAN
    key = f"eig_{which}" if f"eig_{which}" in d else (
        "eig_constrained" if "eig_constrained" in d else None)
    if key is None:
        return
    rel = np.asarray(d["rel_L"], float)
    E = np.asarray(d[key], float)                      # (n_ckpt, n_dof) ascending
    o = np.argsort(rel)
    for m in range(min(n_modes, E.shape[1])):
        ax.plot(rel[o], E[o, m], color="0.35", lw=0.8, alpha=0.75)
    m0 = 0 if which == "constrained" else min(3, E.shape[1] - 1)
    ax.plot(rel[o], E[o, m0], color=C.BEFORE_AFTER_COLORS[1], lw=1.8,
            label=r"lowest non-trivial $\lambda_0$")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.invert_xaxis()
    ax.set_xlabel(r"$L/L_{\rm init}$")
    ax.set_ylabel(rf"rest elastic-Hessian $\lambda_k$ ({which})")
    ax.legend(fontsize=8, loc="lower left")

plot_2a(axes[0, 0], get("allo", "fig2a")); axes[0, 0].set_title("a - rest elastic spectrum vs training")

# ---- 2b / 2f : constrained elastic Hessian vs IMPOSED STRAIN, before vs best-loss ----
# ONE actuation trajectory only -- the deepest subtask (its ramp already spans the
# shallower subtask's strain range). x-axis = actual imposed strain magnitude.
# Two curve sets: grey = initial K, red = best-loss K. Dotted line = where the
# shallow subtask's ramp ends. Accepts the new single-trajectory npz schema
# (before/best/*_strain) and falls back to the old per-subtask schema (sub0_*).
def _fig2b_series(d):
    """[(tag, x_strain, S), ...] + shallow-subtask strain magnitude, either schema."""
    if "before" in d or "best" in d:
        ss = d.get("strain_shallow")
        out = [(t, np.abs(np.asarray(d[f"{t}_strain"], float)), np.asarray(d[t], float))
               for t in ("before", "best") if t in d and f"{t}_strain" in d]
        return out, (abs(float(ss)) if ss is not None and np.isfinite(ss) else None)
    if "sub0_best" in d:                              # old schema: deepest subtask = sub0
        xf = np.asarray(d["sub0_strainfrac"], float)
        depth = abs(float(d["sub0_strain"])) if "sub0_strain" in d else 1.0
        out = [(t, xf * depth, np.asarray(d[f"sub0_{t}"], float))
               for t in ("before", "best") if f"sub0_{t}" in d]
        ss = abs(float(d["sub1_strain"])) if "sub1_strain" in d else None
        return out, ss
    return [], None

def plot_2b(ax, d, n_modes=8, strain_sym=r"|\varepsilon_{\rm in}|"):
    if d is None:
        return
    series, ss = _fig2b_series(d)
    if not series:
        ax.text(0.5, 0.5, "re-run calc fig2b", ha="center", va="center",
                transform=ax.transAxes, color="gray"); return
    cmap = {"before": C.BEFORE_AFTER_COLORS[0], "best": C.BEFORE_AFTER_COLORS[1]}
    have_before = any(t == "before" for t, _, _ in series)
    for tag, x, S in series:
        o = np.argsort(x)
        for m in range(min(n_modes, S.shape[1])):
            ax.plot(x[o], S[o, m], color=cmap[tag], lw=1, alpha=0.85)
    if ss is not None:
        ax.axvline(ss, color="0.45", lw=0.9, ls=":")
    ax.set_yscale("log")
    ax.set_xlabel(rf"imposed strain  ${strain_sym}$")
    ax.set_ylabel(r"constrained elastic $\lambda_k$")
    h = ([Line2D([0], [0], color=C.BEFORE_AFTER_COLORS[0])] if have_before else []) + \
        [Line2D([0], [0], color=C.BEFORE_AFTER_COLORS[1])]
    l = (["before"] if have_before else []) + ["best loss"]
    if ss is not None:
        h.append(Line2D([0], [0], color="0.45", ls=":")); l.append("shallow subtask end")
    ax.legend(h, l, fontsize=8, title="training")
plot_2b(axes[0, 1], get("allo", "fig2b")); axes[0, 1].set_title("b - elastic spectrum vs imposed strain")

# ---- 2c / 2g : STACKED histogram of  lambda0(after) / lambda0(before)  at 3 compressions ----
def plot_2c(ax, d, strain_sym=r"\varepsilon_{\rm in}"):
    if d is None:
        return
    nocomp_key = f"ratio_nocomp_{NOCOMP_HESSIAN}"
    if nocomp_key not in d:
        ax.text(0.5, 0.5, "stale fig2c npz - re-run calc", ha="center", va="center",
                transform=ax.transAxes, color="gray")
        return
    s1 = _strain_label(d.get("strain_sub1"), strain_sym) or "subtask 1"
    s2 = _strain_label(d.get("strain_sub2"), strain_sym) or "subtask 2"
    series = [(rf"no compression (${strain_sym} = 0$)", d[nocomp_key], "#7f7f7f"),
              (s1, d["ratio_sub1"], C.SUBTASK_COLORS[0]),
              (s2, d["ratio_sub2"], C.SUBTASK_COLORS[1])]
    vv = [(lab, np.asarray(v, float), col) for lab, v, col in series]
    vv = [(lab, v[np.isfinite(v) & (v > 0)], col) for lab, v, col in vv]
    vv = [(lab, v, col) for lab, v, col in vv if len(v)]
    if not vv:
        return
    allv = np.concatenate([v for _, v, _ in vv])
    lo, hi = np.nanpercentile(allv, [1, 99])
    bins = np.logspace(np.log10(max(lo, 1e-4)), np.log10(max(hi, lo * 1.1)), 30)
    ax.hist([v for _, v, _ in vv], bins=bins, histtype="barstacked",
            color=[c for _, _, c in vv],
            label=[f"{lab} (n={len(v)})" for lab, v, _ in vv],
            edgecolor="white", linewidth=0.3)
    ax.axvline(1.0, color="k", lw=1.2, ls="--")
    ax.set_xscale("log")
    ax.set_xlabel(r"$\lambda_0^{\rm after} / \lambda_0^{\rm before}$")
    ax.set_ylabel("count (stacked)"); ax.legend(fontsize=7)
plot_2c(axes[0, 2], get("allo", "fig2c_targeted")); axes[0, 2].set_title(r"c - $\lambda_0$ ratio after/before")

# ---- 2d / 2h : where along the trajectory avoided crossings occur ----
def plot_2d(ax, d, n_bins=20):
    if d is None:
        return
    pb = np.asarray(d["pos_before"], float) if "pos_before" in d else np.array([])
    pa = np.asarray(d["pos_after"], float) if "pos_after" in d else np.array([])
    pb = pb[np.isfinite(pb)]; pa = pa[np.isfinite(pa)]
    bins = np.linspace(0, 1, n_bins + 1)
    if len(pb):
        ax.hist(pb, bins=bins, histtype="step", lw=1.6, color=C.BEFORE_AFTER_COLORS[0],
                label=f"before ({len(pb)} events)")
    if len(pa):
        ax.hist(pa, bins=bins, histtype="step", lw=1.6, color=C.BEFORE_AFTER_COLORS[1],
                label=f"after ({len(pa)} events)")
    ax.set_xlabel("actuation-trajectory position  (t+0.5)/(T-1)")
    ax.set_ylabel("# avoided crossings (diabatic)")
    nr = int(d["n_real"]) if "n_real" in d else None
    if len(pb) or len(pa):
        ax.legend(fontsize=8, title=None if nr is None else f"pool: {nr} realizations")
    else:
        ax.text(0.5, 0.5, "no avoided crossings detected", ha="center", va="center",
                transform=ax.transAxes, color="gray")
plot_2d(axes[0, 3], get("allo", "fig2d_targeted")); axes[0, 3].set_title("d - avoided-crossing positions")

# ---- e-h : auxetic mirror ----
plot_2a(axes[1, 0], get("aux", "fig2a")); axes[1, 0].set_title("e - auxetic rest elastic spectrum vs training")
plot_2b(axes[1, 1], get("aux", "fig2b"), strain_sym=r"|\varepsilon_{yy}|"); axes[1, 1].set_title("f - auxetic elastic spectrum vs |compression|")
plot_2c(axes[1, 2], get("aux", "fig2c_targeted"), strain_sym=r"\varepsilon_{yy}"); axes[1, 2].set_title(r"g - auxetic $\lambda_0$ ratio after/before")
plot_2d(axes[1, 3], get("aux", "fig2d_targeted")); axes[1, 3].set_title("h - auxetic avoided-crossing positions")

fig.tight_layout()
savefig(fig, "fig2_aug26")
plt.show()

## Figure 3 — cost-Hessian top modes vs shift susceptibility

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.suptitle("Figure 3 — cost-Hessian top mode vs shift susceptibility "
             "(b = allosteric GENERAL pool, d = auxetic pool)")

def plot_3a(ax, d, keys=("v_mse1", "v_mse2")):
    if d is None:
        return
    for key, col, lab in zip(keys, C.SUBTASK_COLORS, ("subtask 1", "subtask 2")):
        if key not in d:                             # single-subtask task
            continue
        ax.scatter(d["s_shift"], d[key], s=12, alpha=0.6, color=col, label=lab)
    ax.set_xlabel(r"$|s_{\rm shift}|$ per edge"); ax.set_ylabel(r"$|v_{\rm top}|$ per edge")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.legend(fontsize=8)

def plot_3b(ax, d, pool_label=""):
    if d is None:
        return
    r = np.asarray(d["rho"], float); r = r[np.isfinite(r)]
    if len(r) == 0:
        ax.text(0.5, 0.5, "no rho values", ha="center", va="center", transform=ax.transAxes,
                color="gray"); return
    ax.hist(r, bins=np.linspace(-1, 1, 30), color="#4c72b0", alpha=0.8)
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.axvline(np.median(r), color="#c44e52", lw=1.4, label=f"median = {np.median(r):.2f}")
    ax.set_xlabel(r"Spearman $\rho(|v_{\rm top}|, |s_{\rm shift}|)$"); ax.set_ylabel("count")
    nreal = int(d["n_real"]) if "n_real" in d else None
    extra = f", {nreal} realizations" if nreal is not None else ""
    ax.set_title(f"{pool_label}n = {len(r)} (rho per realization x subtask{extra})", fontsize=9)
    ax.legend(fontsize=8)

plot_3a(axes[0, 0], get("allo", "fig3a_allo")); axes[0, 0].set_title("a — allosteric (representative task)")
plot_3b(axes[0, 1], get("allo", "fig3b_general"), pool_label="allosteric general — ")
axes[0, 1].set_title("b — allosteric $\\rho$ distribution (general pool)")
plot_3a(axes[1, 0], get("aux", "fig3a_aux"), keys=("v_sub1", "v_sub2")); axes[1, 0].set_title("c — auxetic (representative task)")
plot_3b(axes[1, 1], get("aux", "fig3b_targeted"), pool_label="auxetic (sweep-served subset) — ")
axes[1, 1].set_title("d — auxetic $\\rho$ distribution (realizations with a sweep)")

fig.tight_layout()
savefig(fig, "fig3_aug26")
plt.show()


## Figure 4 — Fig3Maker panels (one allosteric + one auxetic task, one figure)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle("Figure 4 — spectrum vs strain, incremental-displacement / mode overlaps, network")

d = get("allo", "fig4_allo")
if d is not None:
    ax = axes[0, 0]
    S = d["spectrum"]
    for m in range(S.shape[1]):
        ax.plot(d["eps_in"], S[:, m], lw=0.8, color="tab:blue", alpha=0.6)
    ax.set_yscale("log"); ax.set_xlabel(r"$\varepsilon_{\rm in}$"); ax.set_ylabel(r"$\lambda_k$")
    ax.set_title("a — allosteric elastic spectrum vs input strain")

    ax = axes[0, 1]
    O = d["overlaps"]                       # (T-1, M)
    im = ax.imshow(O.T, aspect="auto", origin="lower",
                   extent=[d["eps_in"][1], d["eps_in"][-1], 0, O.shape[1]], cmap="magma")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=r"$|\langle du|v_k\rangle|$")
    ax.set_xlabel(r"$\varepsilon_{\rm in}$"); ax.set_ylabel("mode index")
    ax.set_title("b — incremental-displacement / mode overlap")

    ax = axes[0, 2]
    lc = C.draw_network(ax, d["nodes"], d["edges"], d["stiffnesses"], cmap="magma")
    fig.colorbar(lc, ax=ax, fraction=0.046, pad=0.02, label=r"$\log_{10} k$")
    ax.set_title("c — allosteric network")

d = get("aux", "fig4_aux")
if d is not None:
    xa = np.abs(np.asarray(d["eps_yy"], float))

    ax = axes[1, 0]
    S = d["spectrum"]
    for m in range(S.shape[1]):
        ax.plot(xa, S[:, m], lw=0.8, color="tab:green", alpha=0.6)
    ax.set_yscale("log"); ax.set_xlabel(r"$|\varepsilon_{yy}|$"); ax.set_ylabel(r"$\lambda_k$")
    ax.set_title(r"d — auxetic elastic spectrum vs $|\varepsilon_{yy}|$")

    ax = axes[1, 1]
    if "overlaps" in d:
        O = d["overlaps"]                  # (T-1, M)
        im = ax.imshow(O.T, aspect="auto", origin="lower",
                       extent=[xa[1], xa[-1], 0, O.shape[1]], cmap="magma")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label=r"$|\langle du|v_k\rangle|$")
        ax.set_xlabel(r"$|\varepsilon_{yy}|$"); ax.set_ylabel("mode index")
        ax.set_title("e — incremental-displacement / mode overlap (auxetic)")
    else:
        ax.text(0.5, 0.5, "re-run calc fig4_aux for overlaps", ha="center", va="center",
                transform=ax.transAxes, color="gray")
        ax.set_axis_off()

    ax = axes[1, 2]
    lc = C.draw_network(ax, d["nodes"], d["edges"], d["stiffnesses"], cmap="magma")
    fig.colorbar(lc, ax=ax, fraction=0.046, pad=0.02, label=r"$\log_{10} k$")
    ax.set_title("f — auxetic network")

fig.tight_layout()
savefig(fig, "fig4_aug26")
plt.show()

## Figure 5 — susceptibility components vs mechanical quantities

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle(f"Figure 5 — POOL={POOL}")

def scatter_pair(ax, d, xkey, ykey, xlabel, ylabel):
    if d is None:
        return
    for subtask, col in zip((0, 1), C.SUBTASK_COLORS):
        if f"sub{subtask}_{xkey}" not in d:          # task may have a single subtask
            continue
        x = np.abs(d[f"sub{subtask}_{xkey}"]); y = np.abs(d[f"sub{subtask}_{ykey}"])
        ax.scatter(x, y, s=12, alpha=0.6, color=col, label=f"subtask {subtask + 1}")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.legend(fontsize=8)

da = get("allo", "fig5abc_allo")
scatter_pair(axes[0, 0], da, "strain", "s_par",  r"$|\varepsilon_i|$", r"$s_i^\parallel$")
scatter_pair(axes[0, 1], da, "stress", "s_perp", r"$\sigma_i$",        r"$s_i^\perp$")
scatter_pair(axes[0, 2], da, "stress", "s_eq",   r"$\sigma_i$",        r"$s_i^{\rm eq}$")
axes[0, 0].set_title("a"); axes[0, 1].set_title("b"); axes[0, 2].set_title("c")

def heatmap(ax, d):
    if d is None:
        return
    M = d["matrix"].astype(float)
    im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(d["cols"]))); ax.set_xticklabels(d["cols"], rotation=30, ha="right")
    ax.set_yticks(range(len(d["rows"]))); ax.set_yticklabels([f"|{r}|" for r in d["rows"]])
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="mean Spearman $\\rho$")
heatmap(axes[0, 3], get("allo", f"fig5d_{POOL}")); axes[0, 3].set_title("d — Spearman heatmap")

dx = get("aux", "fig5abc_aux")
scatter_pair(axes[1, 0], dx, "strain", "s_par",  r"$|\varepsilon_i|$", r"$s_i^\parallel$")
scatter_pair(axes[1, 1], dx, "stress", "s_perp", r"$\sigma_i$",        r"$s_i^\perp$")
scatter_pair(axes[1, 2], dx, "stress", "s_eq",   r"$\sigma_i$",        r"$s_i^{\rm eq}$")
axes[1, 0].set_title("e"); axes[1, 1].set_title("f"); axes[1, 2].set_title("g")
heatmap(axes[1, 3], get("aux", f"fig5d_{POOL}")); axes[1, 3].set_title("h — Spearman heatmap (aux)")

fig.tight_layout()
savefig(fig, "fig5_aug26")
plt.show()

## Debug — per-task loss distribution (targeted pool)

Sanity check that the networks feeding Figures 1–5 actually converged. For every
**targeted** task shown in Fig 1 (allosteric `geometry_targeted`, auxetic
`auxetic_nets_aug/targeted`), read the training loss trajectory of **all**
realizations on disk and plot the distribution of

* final loss  (last step of the combined `(mse1+mse2)/2` for allosteric,
  mean-over-subtasks `loss_trajectory` for auxetic), and
* running-min loss.

The ✕ marks `realization 0` — the one actually drawn in Fig 1a/1b/1d. Pure disk
I/O, no physics.

In [ ]:
import json as _json

DEBUG_TASKS = list(range(5))          # targeted tasks 0..4 (as in Fig 1)

# Which realization each panel of Fig 1 actually draws, per task:
#   auxetic  -> realization 0 (REP_AUX_TASKS = [(t, 0)])
#   allosteric Fig 1b -> first *converged* realization of the task in geometry_targeted
def _allo_fig1_realization(tid):
    for (g, t, r) in C.discover_allosteric("targeted", require_converged=True):
        if t == tid:
            return r
    return None

def _allo_task_losses(tid):
    """(rid, final_loss, min_loss, ratio) per realization for geometry_targeted/task_tid."""
    tdir = C.ALLO_DATA_DIR / "geometry_targeted" / f"task_{tid}"
    rows = []
    if not tdir.is_dir():
        return rows
    for rdir in sorted(tdir.glob("realization_*"), key=lambda p: int(p.name.split("_")[1])):
        f1, f2 = rdir / "mse1.npy", rdir / "mse2.npy"
        if not (f1.exists() and f2.exists()):
            continue
        c = C.combined_loss(np.load(f1), np.load(f2))
        if len(c) == 0 or c[0] <= 0:
            continue
        rows.append((int(rdir.name.split("_")[1]), float(c[-1]), float(np.nanmin(c)),
                     float(np.nanmin(c) / c[0])))
    return rows

def _aux_task_losses(tid):
    tdir = C.AUX_DATA_DIR / f"task_{tid:02d}"
    rows = []
    if not tdir.is_dir():
        return rows
    for rdir in sorted(tdir.glob("realization_*"), key=lambda p: int(p.name.split("_")[1])):
        lt = C._auxfile(rdir, "loss_trajectory", "npy")
        if not lt.exists():
            continue
        L = np.asarray(np.load(lt), dtype=float)
        s = L.mean(axis=1) if L.ndim > 1 else L
        if len(s) == 0 or s[0] <= 0:
            continue
        rows.append((int(rdir.name.split("_")[1]), float(s[-1]), float(np.nanmin(s)),
                     float(np.nanmin(s) / s[0])))
    return rows

_allo_pick = {t: _allo_fig1_realization(t) for t in DEBUG_TASKS}
_aux_pick = {t: 0 for t in DEBUG_TASKS}
print("Fig 1 realization per task  — allo:", _allo_pick, " aux:", _aux_pick)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
fig.suptitle("Debug — targeted-pool training loss per task (all realizations on disk)")

for ax, (label, loader, pick) in zip(
        axes,
        [("allosteric  (geometry_targeted)", _allo_task_losses, _allo_pick),
         ("auxetic  (auxetic_nets_aug/targeted)", _aux_task_losses, _aux_pick)]):
    any_rows = False
    for tid in DEBUG_TASKS:
        rows = loader(tid)
        if not rows:
            print(f"  [{label}] task {tid}: no realizations found")
            continue
        any_rows = True
        rid = np.array([r[0] for r in rows])
        final = np.array([r[1] for r in rows])
        mn = np.array([r[2] for r in rows])
        jit = (np.random.default_rng(tid).random(len(rows)) - 0.5) * 0.28
        ax.scatter(tid + jit, final, s=26, color="#4c72b0", alpha=0.75,
                   label="final loss" if tid == DEBUG_TASKS[0] else None)
        ax.scatter(tid + jit, mn, s=16, color="#dd8452", alpha=0.7, marker="s",
                   label="running-min loss" if tid == DEBUG_TASKS[0] else None)
        pr = pick.get(tid)
        if pr is not None and pr in rid:
            k0 = int(np.where(rid == pr)[0][0])
            ax.scatter([tid], [final[k0]], s=150, marker="x", color="k", zorder=6,
                       label="realization drawn in Fig 1" if tid == DEBUG_TASKS[0] else None)
        med = np.median(final)
        ax.hlines(med, tid - 0.32, tid + 0.32, color="#4c72b0", lw=2)
        print(f"  [{label}] task {tid}: n={len(rows):2d}  fig1_real={pr}  "
              f"final median={med:.2e}  best={mn.min():.2e}  worst-final={final.max():.2e}")
    ax.set_yscale("log")
    ax.set_xlabel("task id"); ax.set_title(label)
    ax.set_xticks(DEBUG_TASKS)
    if any_rows:
        ax.legend(fontsize=8, loc="best")
axes[0].set_ylabel("training loss")
fig.tight_layout()
savefig(fig, "figD_targeted_loss_dist")
plt.show()